# Rebuild Voyage Click Outputs from Precise Positions

This notebook rebuilds all Pulse_*.wav, Waveform_*.png, and Analysis_*.png artifacts from pulse_precise_positions.txt. It is checkpointed, resumable, validates all outputs before deleting stale Click directories, and regenerates click_count_by_voyage.png.

Kernel: D:\Python_env\toothwhale\python.exe


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from tqdm import tqdm


BASE_DIR = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
LOCATE_DIR = BASE_DIR / "09_Locate"
SOURCE_DIR = BASE_DIR / "00_Data" / "ClickTrains"
REPORT_DIR = LOCATE_DIR / "Locate_Batch_Report"
POSITION_PATH = REPORT_DIR / "pulse_precise_positions.txt"
CHECKPOINT_PATH = REPORT_DIR / "regeneration_checkpoint.jsonl"
RUN_MANIFEST_PATH = REPORT_DIR / "regeneration_run_manifest.json"
DELETE_MANIFEST_PATH = REPORT_DIR / "regeneration_delete_manifest.json"
SUMMARY_PATH = REPORT_DIR / "regeneration_summary.json"
COUNT_PLOT_PATH = LOCATE_DIR / "click_count_by_voyage.png"

NOTEBOOK_VERSION = "1.0.0"
EXPECTED_SAMPLE_RATE = 576_000
HALF_WINDOW_S = 5e-3
EXPECTED_SEGMENT_SAMPLES = 5_761
EXPECTED_POSITION_ROWS = 18_301
EXPECTED_INITIAL_STALE_DIRS = 119
PNG_MIN_BYTES = 1_024
CLICK_DIR_RE = re.compile(r"^Click_\d{3}$")
PULSE_TRAIN_RE = re.compile(r"^PulseTrain_\d{3}$")
VOYAGE_RE = re.compile(r"^Voyage_Ori_(?:\d{2}|\d{2}-\d{2})$")


def atomic_write_text(path: Path, text: str) -> None:
    """Write UTF-8 text atomically in the destination directory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_positions() -> list[dict[str, object]]:
    """Read and fully validate the precise-position table."""
    if not POSITION_PATH.exists():
        raise FileNotFoundError(f"Position file not found: {POSITION_PATH}")
    with POSITION_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
        data_lines = [line for line in handle if line.strip() and not line.startswith("#")]
    reader = csv.DictReader(data_lines, delimiter="\t")
    required = {
        "Voyage_Ori",
        "PulseTrain",
        "PulseNumber",
        "PositionSample_1based",
        "SampleRate_Hz",
    }
    missing = required.difference(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Position table missing columns: {sorted(missing)}")

    rows: list[dict[str, object]] = []
    keys: set[str] = set()
    for source_row in reader:
        voyage = source_row["Voyage_Ori"].strip()
        pulse_train = source_row["PulseTrain"].strip()
        pulse_number = int(source_row["PulseNumber"])
        position_1based = int(source_row["PositionSample_1based"])
        sample_rate = int(source_row["SampleRate_Hz"])
        if not VOYAGE_RE.fullmatch(voyage):
            raise ValueError(f"Invalid voyage name: {voyage}")
        if not PULSE_TRAIN_RE.fullmatch(pulse_train):
            raise ValueError(f"Invalid PulseTrain name: {pulse_train}")
        if pulse_number <= 0 or position_1based <= 0:
            raise ValueError(f"Invalid pulse row: {source_row}")
        if sample_rate != EXPECTED_SAMPLE_RATE:
            raise ValueError(
                f"Unexpected sample rate {sample_rate} for {pulse_train}/Pulse_{pulse_number}"
            )
        click_name = f"Click_{pulse_number:03d}"
        key = f"{voyage}/{pulse_train}/{click_name}"
        if key in keys:
            raise ValueError(f"Duplicate position key: {key}")
        keys.add(key)
        click_dir = LOCATE_DIR / voyage / pulse_train / click_name
        rows.append(
            {
                "key": key,
                "voyage": voyage,
                "pulse_train": pulse_train,
                "pulse_number": pulse_number,
                "position_1based": position_1based,
                "sample_rate": sample_rate,
                "click_dir": click_dir,
                "wav_path": click_dir / f"Pulse_{pulse_number:03d}.wav",
                "waveform_path": click_dir / f"Waveform_{pulse_number:03d}.png",
                "analysis_path": click_dir / f"Analysis_{pulse_number:03d}.png",
            }
        )
    if len(rows) != EXPECTED_POSITION_ROWS:
        raise ValueError(
            f"Expected {EXPECTED_POSITION_ROWS} position rows, found {len(rows)}"
        )
    rows.sort(key=lambda row: (str(row["pulse_train"]), int(row["pulse_number"])))
    return rows


def build_input_signature(rows: list[dict[str, object]]) -> tuple[str, dict[str, dict[str, int]]]:
    """Fingerprint the position table plus all source PulseTrain WAV metadata."""
    source_signature: dict[str, dict[str, int]] = {}
    for pulse_train in sorted({str(row["pulse_train"]) for row in rows}):
        source_path = SOURCE_DIR / pulse_train / "PulseTrain.wav"
        if not source_path.exists():
            raise FileNotFoundError(f"Source WAV not found: {source_path}")
        stat = source_path.stat()
        info = sf.info(str(source_path))
        if info.samplerate != EXPECTED_SAMPLE_RATE or info.channels != 1:
            raise ValueError(f"Unexpected source WAV format: {source_path}\n{info}")
        source_signature[pulse_train] = {
            "size": int(stat.st_size),
            "mtime_ns": int(stat.st_mtime_ns),
            "frames": int(info.frames),
            "sample_rate": int(info.samplerate),
        }

    for row in rows:
        source_info = source_signature[str(row["pulse_train"])]
        if int(row["position_1based"]) > source_info["frames"]:
            raise ValueError(
                f"Position outside source WAV: {row['key']} position={row['position_1based']} "
                f"frames={source_info['frames']}"
            )

    payload = {
        "notebook_version": NOTEBOOK_VERSION,
        "position_sha256": sha256_file(POSITION_PATH),
        "sources": source_signature,
    }
    signature = hashlib.sha256(
        json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()
    return signature, source_signature


def load_completed_keys(input_signature: str) -> set[str]:
    """Load only checkpoint entries created for the current inputs."""
    completed: set[str] = set()
    if not CHECKPOINT_PATH.exists():
        return completed
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if record.get("input_signature") == input_signature and record.get("status") == "ok":
                completed.add(str(record["key"]))
    return completed


def output_is_valid(row: dict[str, object]) -> bool:
    """Perform inexpensive but strict validation for one output trio."""
    wav_path = Path(row["wav_path"])
    waveform_path = Path(row["waveform_path"])
    analysis_path = Path(row["analysis_path"])
    if not wav_path.exists() or not waveform_path.exists() or not analysis_path.exists():
        return False
    if waveform_path.stat().st_size < PNG_MIN_BYTES or analysis_path.stat().st_size < PNG_MIN_BYTES:
        return False
    try:
        info = sf.info(str(wav_path))
    except RuntimeError:
        return False
    return bool(
        info.samplerate == EXPECTED_SAMPLE_RATE
        and info.channels == 1
        and info.frames == EXPECTED_SEGMENT_SAMPLES
        and info.subtype == "PCM_16"
    )


def extract_segment(signal: np.ndarray, center_idx: int, half_win: int) -> np.ndarray:
    """Reuse the old Locate behavior: inclusive +/-5 ms with zero padding."""
    segment_length = 2 * half_win + 1
    start = center_idx - half_win
    end = center_idx + half_win + 1
    actual_start = max(0, start)
    actual_end = min(len(signal), end)
    pad_left = max(0, -start)
    pad_right = max(0, end - len(signal))
    segment = np.concatenate(
        [
            np.zeros(pad_left, dtype=signal.dtype),
            signal[actual_start:actual_end],
            np.zeros(pad_right, dtype=signal.dtype),
        ]
    )
    if len(segment) < segment_length:
        segment = np.pad(segment, (0, segment_length - len(segment)))
    elif len(segment) > segment_length:
        segment = segment[:segment_length]
    if len(segment) != EXPECTED_SEGMENT_SAMPLES:
        raise AssertionError(f"Unexpected segment length: {len(segment)}")
    return segment


def save_waveform_figure(segment: np.ndarray, sample_rate: int, label: str, path: Path) -> None:
    """Reuse the committed pre-modification Waveform PNG style."""
    half_win = len(segment) // 2
    time_ms = np.arange(-half_win, half_win + 1) / sample_rate * 1000.0
    temporary = path.with_name(path.stem + ".tmp.png")
    fig, axis = plt.subplots(figsize=(6, 3))
    try:
        axis.plot(time_ms, segment, linewidth=0.8, color="#333333")
        axis.axvline(x=0, color="red", linestyle="--", alpha=0.6, linewidth=0.8)
        axis.set_title(f"{label} — Waveform")
        axis.set_xlabel("Relative Time (ms)")
        axis.set_ylabel("Amplitude")
        axis.set_xlim(-5, 5)
        axis.grid(True, alpha=0.3)
        maximum = float(np.max(np.abs(segment))) * 1.1
        axis.set_ylim(-maximum if maximum else -1.0, maximum if maximum else 1.0)
        fig.tight_layout()
        fig.savefig(temporary, dpi=150)
    finally:
        plt.close(fig)
    os.replace(temporary, path)


def save_analysis_figure(segment: np.ndarray, sample_rate: int, label: str, path: Path) -> None:
    """Reuse the committed pre-modification TEO/autocorrelation PNG style."""
    half_win = len(segment) // 2
    time_ms = np.arange(-half_win, half_win + 1) / sample_rate * 1000.0
    teo_raw = np.zeros_like(segment)
    teo_raw[1:-1] = segment[1:-1] ** 2 - segment[:-2] * segment[2:]
    autocorrelation = np.correlate(segment, segment, mode="full")
    zero_lag = autocorrelation[len(autocorrelation) // 2]
    if zero_lag != 0:
        autocorrelation = autocorrelation / zero_lag
    lags = np.arange(len(autocorrelation)) - len(autocorrelation) // 2
    lags_ms = lags / sample_rate * 1000.0

    temporary = path.with_name(path.stem + ".tmp.png")
    fig, (axis_teo, axis_acf) = plt.subplots(2, 1, figsize=(7, 5))
    try:
        axis_teo.plot(time_ms, teo_raw, color="#D95319", linewidth=1.0)
        axis_teo.set_title(f"{label} — Teager Energy Operator")
        axis_teo.set_xlabel("Time (ms)")
        axis_teo.set_ylabel("TEO Energy")
        axis_teo.set_xlim(-5, 5)
        axis_teo.grid(True, alpha=0.3)
        axis_acf.plot(lags_ms, autocorrelation, color="#0072BD", linewidth=1.0)
        axis_acf.set_title(f"{label} — Autocorrelation")
        axis_acf.set_xlabel("Delay (ms)")
        axis_acf.set_ylabel("Correlation Coeff")
        axis_acf.set_xlim(-5, 5)
        axis_acf.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(temporary, dpi=150)
    finally:
        plt.close(fig)
    os.replace(temporary, path)


def write_output_trio(row: dict[str, object], signal: np.ndarray) -> None:
    click_dir = Path(row["click_dir"])
    click_dir.mkdir(parents=True, exist_ok=True)
    center_idx = int(row["position_1based"]) - 1
    half_win = round(HALF_WINDOW_S * int(row["sample_rate"]))
    segment = extract_segment(signal, center_idx, half_win)
    wav_path = Path(row["wav_path"])
    temporary_wav = wav_path.with_name(wav_path.stem + ".tmp.wav")
    sf.write(
        str(temporary_wav),
        segment,
        int(row["sample_rate"]),
        format="WAV",
        subtype="PCM_16",
    )
    os.replace(temporary_wav, wav_path)
    label = f"{row['pulse_train']} / Click_{int(row['pulse_number']):03d}"
    save_waveform_figure(segment, int(row["sample_rate"]), label, Path(row["waveform_path"]))
    save_analysis_figure(segment, int(row["sample_rate"]), label, Path(row["analysis_path"]))
    if not output_is_valid(row):
        raise RuntimeError(f"Output validation failed immediately after writing: {row['key']}")


def regenerate(rows: list[dict[str, object]], input_signature: str) -> dict[str, int]:
    """Overwrite every item on a fresh run; resume only verified checkpoint items."""
    completed = load_completed_keys(input_signature)
    checkpoint_mode = "a" if CHECKPOINT_PATH.exists() else "w"
    counters = Counter(generated=0, resumed=0, failed=0)
    current_train = ""
    current_signal: np.ndarray | None = None
    with CHECKPOINT_PATH.open(checkpoint_mode, encoding="utf-8", buffering=1) as checkpoint:
        progress = tqdm(rows, total=len(rows), desc="总体重建进度", unit="click", dynamic_ncols=True)
        for index, row in enumerate(progress, start=1):
            key = str(row["key"])
            if key in completed and output_is_valid(row):
                counters["resumed"] += 1
                continue
            pulse_train = str(row["pulse_train"])
            if pulse_train != current_train:
                source_path = SOURCE_DIR / pulse_train / "PulseTrain.wav"
                current_signal, sample_rate = sf.read(str(source_path), dtype="float64")
                if current_signal.ndim != 1 or sample_rate != EXPECTED_SAMPLE_RATE:
                    raise ValueError(f"Unexpected source WAV at runtime: {source_path}")
                current_train = pulse_train
                progress.set_postfix(train=pulse_train, generated=counters["generated"])
            try:
                assert current_signal is not None
                write_output_trio(row, current_signal)
                record = {
                    "input_signature": input_signature,
                    "key": key,
                    "status": "ok",
                    "completed_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
                }
                checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
                checkpoint.flush()
                if index % 25 == 0:
                    os.fsync(checkpoint.fileno())
                counters["generated"] += 1
            except Exception:
                counters["failed"] += 1
                raise
    return {key: int(value) for key, value in counters.items()}


def validate_all_outputs(rows: list[dict[str, object]]) -> None:
    failures: list[str] = []
    for row in tqdm(rows, desc="逐文件验收", unit="click", dynamic_ncols=True):
        if not output_is_valid(row):
            failures.append(str(row["key"]))
    if failures:
        raise RuntimeError(f"{len(failures)} output trios failed validation; first: {failures[:10]}")

    # Decode a deterministic sample from every voyage for actual PNG readability.
    first_by_voyage: dict[str, dict[str, object]] = {}
    for row in rows:
        first_by_voyage.setdefault(str(row["voyage"]), row)
    for row in first_by_voyage.values():
        for path_key in ("waveform_path", "analysis_path"):
            image = mpimg.imread(Path(row[path_key]))
            if image.size == 0:
                raise RuntimeError(f"Decoded empty PNG: {row[path_key]}")


def create_or_load_delete_manifest(expected_wavs: set[Path]) -> list[Path]:
    """Create an atomic first-run deletion list, or resume an existing one."""
    if DELETE_MANIFEST_PATH.exists():
        manifest = json.loads(DELETE_MANIFEST_PATH.read_text(encoding="utf-8"))
        return [Path(item) for item in manifest["stale_click_dirs"]]

    stale_dirs: set[Path] = set()
    for voyage_dir in LOCATE_DIR.glob("Voyage_Ori_*"):
        if not voyage_dir.is_dir():
            continue
        for wav_path in voyage_dir.glob("PulseTrain_*/Click_*/Pulse_*.wav"):
            if wav_path.resolve() not in expected_wavs:
                stale_dirs.add(wav_path.parent.resolve())
    stale = sorted(stale_dirs, key=lambda path: str(path).lower())
    if len(stale) != EXPECTED_INITIAL_STALE_DIRS:
        raise RuntimeError(
            f"Refusing deletion: expected {EXPECTED_INITIAL_STALE_DIRS} stale Click directories, "
            f"found {len(stale)}"
        )
    for click_dir in stale:
        relative = click_dir.relative_to(LOCATE_DIR)
        parts = relative.parts
        if (
            len(parts) != 3
            or not VOYAGE_RE.fullmatch(parts[0])
            or not PULSE_TRAIN_RE.fullmatch(parts[1])
            or not CLICK_DIR_RE.fullmatch(parts[2])
        ):
            raise RuntimeError(f"Unsafe stale directory path: {click_dir}")
    manifest = {
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "expected_count": EXPECTED_INITIAL_STALE_DIRS,
        "stale_click_dirs": [str(path) for path in stale],
    }
    atomic_write_text(DELETE_MANIFEST_PATH, json.dumps(manifest, ensure_ascii=False, indent=2))
    return stale


def delete_stale_click_dirs(stale: list[Path]) -> int:
    deleted = 0
    for click_dir in tqdm(stale, desc="删除过期Click目录", unit="dir", dynamic_ncols=True):
        if not click_dir.exists():
            continue
        relative = click_dir.resolve().relative_to(LOCATE_DIR.resolve())
        parts = relative.parts
        if (
            len(parts) != 3
            or not VOYAGE_RE.fullmatch(parts[0])
            or not PULSE_TRAIN_RE.fullmatch(parts[1])
            or not CLICK_DIR_RE.fullmatch(parts[2])
        ):
            raise RuntimeError(f"Refusing unsafe recursive deletion: {click_dir}")
        shutil.rmtree(click_dir)
        deleted += 1
    return deleted


def disk_wav_counts(expected_wavs: set[Path]) -> dict[str, int]:
    actual_wavs: set[Path] = set()
    voyage_counts: Counter[str] = Counter()
    for voyage_dir in LOCATE_DIR.glob("Voyage_Ori_*"):
        if not voyage_dir.is_dir():
            continue
        for wav_path in voyage_dir.glob("PulseTrain_*/Click_*/Pulse_*.wav"):
            resolved = wav_path.resolve()
            actual_wavs.add(resolved)
            voyage_counts[voyage_dir.name] += 1
    if actual_wavs != expected_wavs:
        missing = sorted(str(path) for path in expected_wavs - actual_wavs)
        extra = sorted(str(path) for path in actual_wavs - expected_wavs)
        raise RuntimeError(
            f"Final WAV set mismatch: missing={len(missing)}, extra={len(extra)}, "
            f"missing_examples={missing[:5]}, extra_examples={extra[:5]}"
        )
    if sum(voyage_counts.values()) != EXPECTED_POSITION_ROWS:
        raise RuntimeError(f"Final WAV total is {sum(voyage_counts.values())}")
    return dict(voyage_counts)


def voyage_sort_key(name: str) -> tuple[int, str]:
    suffix = name.removeprefix("Voyage_Ori_")
    return int(suffix.split("-", 1)[0]), suffix


def save_count_plot(voyage_counts: dict[str, int]) -> None:
    voyage_names = sorted(
        {path.name for path in LOCATE_DIR.glob("Voyage_Ori_*") if path.is_dir()},
        key=voyage_sort_key,
    )
    labels = [name.removeprefix("Voyage_Ori_") for name in voyage_names]
    counts = [int(voyage_counts.get(name, 0)) for name in voyage_names]
    temporary = COUNT_PLOT_PATH.with_name(COUNT_PLOT_PATH.stem + ".tmp.png")
    fig, axis = plt.subplots(figsize=(16, 8))
    try:
        bars = axis.bar(labels, counts, color="#1f77b4", edgecolor="#1f4e79", alpha=0.85)
        axis.set_title("Click Count by Voyage in 09_Locate", fontsize=18, pad=16)
        axis.set_xlabel("Voyage No.", fontsize=14)
        axis.set_ylabel("Click Count", fontsize=14)
        axis.grid(axis="y", linestyle="--", alpha=0.3)
        axis.tick_params(axis="x", rotation=45, labelsize=11)
        axis.tick_params(axis="y", labelsize=11)
        maximum = max(counts) if counts else 1
        for bar, count in zip(bars, counts):
            axis.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + maximum * 0.012,
                str(count),
                ha="center",
                va="bottom",
                fontsize=10,
            )
        axis.set_ylim(0, maximum * 1.12)
        fig.tight_layout()
        fig.savefig(temporary, dpi=150)
    finally:
        plt.close(fig)
    os.replace(temporary, COUNT_PLOT_PATH)


def main() -> dict[str, object]:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    started_at = time.strftime("%Y-%m-%dT%H:%M:%S%z")
    print("[1/7] 读取并验证精确位置表", flush=True)
    rows = read_positions()
    input_signature, source_signature = build_input_signature(rows)
    position_sha256 = sha256_file(POSITION_PATH)
    manifest = {
        "notebook_version": NOTEBOOK_VERSION,
        "started_at": started_at,
        "input_signature": input_signature,
        "position_sha256": position_sha256,
        "position_rows": len(rows),
        "source_pulse_trains": len(source_signature),
    }
    atomic_write_text(RUN_MANIFEST_PATH, json.dumps(manifest, ensure_ascii=False, indent=2))

    print("[2/7] 覆盖生成 WAV、Waveform PNG 和 Analysis PNG", flush=True)
    generation = regenerate(rows, input_signature)
    print(f"生成统计: {generation}", flush=True)

    print("[3/7] 验收全部18,301组三件套", flush=True)
    validate_all_outputs(rows)

    print("[4/7] 固化并检查119组过期Click删除清单", flush=True)
    expected_wavs = {Path(row["wav_path"]).resolve() for row in rows}
    stale_dirs = create_or_load_delete_manifest(expected_wavs)

    print("[5/7] 直接删除过期Click目录", flush=True)
    deleted_this_run = delete_stale_click_dirs(stale_dirs)

    print("[6/7] 核验最终WAV集合并更新航次计数图", flush=True)
    voyage_counts = disk_wav_counts(expected_wavs)
    save_count_plot(voyage_counts)
    decoded_plot = mpimg.imread(COUNT_PLOT_PATH)
    if decoded_plot.size == 0:
        raise RuntimeError("click_count_by_voyage.png decoded as an empty image")

    print("[7/7] 写出最终摘要", flush=True)
    deleted_total = sum(not path.exists() for path in stale_dirs)
    summary: dict[str, object] = {
        "status": "complete",
        "started_at": started_at,
        "completed_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "notebook_version": NOTEBOOK_VERSION,
        "input_signature": input_signature,
        "position_sha256": position_sha256,
        "position_rows": len(rows),
        "source_pulse_trains": len(source_signature),
        "generation": generation,
        "validated_output_trios": len(rows),
        "stale_click_dirs_in_manifest": len(stale_dirs),
        "stale_click_dirs_deleted_this_run": deleted_this_run,
        "stale_click_dirs_deleted_total": deleted_total,
        "final_wav_count": sum(voyage_counts.values()),
        "voyage_counts": dict(sorted(voyage_counts.items(), key=lambda item: voyage_sort_key(item[0]))),
        "count_plot": str(COUNT_PLOT_PATH),
        "failures": 0,
    }
    if deleted_total != EXPECTED_INITIAL_STALE_DIRS:
        raise RuntimeError(
            f"Deletion incomplete: {deleted_total}/{EXPECTED_INITIAL_STALE_DIRS} stale directories removed"
        )
    atomic_write_text(SUMMARY_PATH, json.dumps(summary, ensure_ascii=False, indent=2))
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
    return summary


SUMMARY = main()
